# Data exploration

Load the Swissgrid 2024 file and extract the end-user consumption of the Swiss control block as a clean time series.

## Available sheets

Read the file. The yearly file contains several sheets. We first list them to find the one with the 15-minute values.

In [13]:
import pandas as pd

path = "../data/raw/EnergieUebersichtCH-2024.xlsx"
# sheet_name=None loads all sheets as a dict {sheet name: DataFrame}
sheets = pd.read_excel(path, sheet_name=None)
print(sheets.keys())

dict_keys(['Einstellungen', 'Datetime', 'Uebersicht', 'Zeitreihen0h15'])


## Load the 15-minute series

Sheet 'Zeitreihen0h15': row 1 contains the column titles, row 2 the units (kWh), the data starts in row 3. Column A holds the timestamps, column B the end-user consumption.

In [14]:
df = pd.read_excel(
    path,
    sheet_name='Zeitreihen0h15',
    skiprows=[1],
    usecols='A:B',
)
# check if loaded correctly
df.head()

,Unnamed: 0,Summe endverbrauchte Energie Regelblock Schweiz\nTotal energy consumed by end users in the Swiss controlblock
0,01.01.2024 00:15,1.443842e+06
1,01.01.2024 00:30,1.442366e+06
2,01.01.2024 00:45,1.432568e+06
3,01.01.2024 01:00,1.420472e+06
4,01.01.2024 01:15,1.445856e+06


In [15]:
# check types of relevant consumption values
df.dtypes

Unnamed: 0                                                                                                           str
Summe endverbrauchte Energie Regelblock Schweiz\nTotal energy consumed by end users in the Swiss controlblock    float64
dtype: object

## Clean up

Short column names, timestamps parsed with an explicit format (day first),
timestamp as index. Each timestamp marks the end of a 15-minute interval.

The timestamps are Swiss local time (CET in winter, CEST in summer). Because of
the daylight saving time changes, one hour of labels is missing in March and one
hour is duplicated in October. We localize the timestamps to Europe/Zurich and
convert them to UTC, which gives a regular 15-minute grid (verified below).

In [16]:
df.columns = ["timestamps", "load"]
# specify the format of the timestamps to avoid warnings about ambiguous times during DST transitions
df["timestamps"] = pd.to_datetime(df["timestamps"], format="%d.%m.%Y %H:%M")
df = df.set_index("timestamps")
# specify the timezone of the timestamps to avoid warnings about ambiguous times during DST transitions
df.index = df.index.tz_localize("Europe/Zurich", ambiguous="infer")
df.index = df.index.tz_convert("UTC")
df.dtypes

load    float64
dtype: object

In [17]:
df.head()

,load
timestamps,
2023-12-31 23:15:00+00:00,1.443842e+06
2023-12-31 23:30:00+00:00,1.442366e+06
2023-12-31 23:45:00+00:00,1.432568e+06
2024-01-01 00:00:00+00:00,1.420472e+06
2024-01-01 00:15:00+00:00,1.445856e+06


## Sanity check

Expected: 2024 is a leap year, so 366 days x 96 quarter hours = 35136 rows.
First timestamp 2023-12-31 23:15 UTC (2024-01-01 00:15 local time), last timestamp
2024-12-31 23:00 UTC (2025-01-01 00:00 local time). Timestamps mark the end of each
interval. No missing values, no duplicates, and all consecutive timestamps 15 minutes
apart (thanks to the conversion to UTC).

In [18]:
print(len(df))
print(df.index.min(), df.index.max())
print(df.isna().sum())
print(df.index.duplicated().sum())
print(df.index.to_series().diff().value_counts())
print(df[df.index.duplicated(keep=False)])

35136
2023-12-31 23:15:00+00:00 2024-12-31 23:00:00+00:00
load    0
dtype: int64
0
timestamps
0 days 00:15:00    35135
Name: count, dtype: int64
Empty DataFrame
Columns: [load]
Index: []


## Units and hourly resolution

Swissgrid reports energy per quarter hour in kWh. We convert it to average power in MW (kWh per 15 min × 4 / 1000), the usual unit for grid load. Then we aggregate to hourly resolution, since we want to forecast hourly load.